# XAI-Based Detection of Spurious Correlations — Experiment Walkthrough

Waterbirds + ResNet18: baseline ERM, Grad-CAM background attribution, RQ2 association analysis, and XAI-guided targeted augmentation.

Run `python run_experiments.py` first to populate `results/`, `checkpoints/` and `figures/`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RESULTS = ROOT / 'results'
FIGURES = ROOT / 'figures'

def show(path, width=12):
    img = plt.imread(path)
    plt.figure(figsize=(width, width * img.shape[0] / img.shape[1]))
    plt.imshow(img); plt.axis('off'); plt.show()

## 1. Dataset and groups

Official split; group = `2 * bird_label + place`.

In [ ]:
meta = pd.read_csv(ROOT / 'data/raw/metadata.csv')
meta['split'] = meta['split'].map({0: 'train', 1: 'val', 2: 'test'})
meta['group'] = 2 * meta['y'] + meta['place']
meta.groupby(['split', 'group']).size().unstack(fill_value=0)

## 2. Baseline (Experiment A)

Training curves, group accuracy, confusion matrix.

In [ ]:
hist = pd.read_csv(RESULTS / 'baseline/history.csv')
ax = hist.plot(x='epoch', y=['train_loss', 'val_loss'], marker='o')
ax.set_title('Baseline loss'); plt.show()
ax = hist.plot(x='epoch', y=['train_acc', 'val_acc', 'val_worst_group_acc'], marker='o')
ax.set_title('Baseline accuracy'); plt.show()

with open(RESULTS / 'baseline/metrics.json') as f:
    base_metrics = json.load(f)
{k: base_metrics[k] for k in ['accuracy', 'f1_macro', 'worst_group_accuracy', 'minority_mean_accuracy', 'group_accuracy']}

In [ ]:
show(FIGURES / 'group_accuracy.png')
show(FIGURES / 'confusion_matrix.png', width=7)

## 3. Grad-CAM (Experiment B / RQ1)

For each test image: heatmap, and background attribution ratio = share of Grad-CAM mass outside the annotated bird box.

In [ ]:
show(FIGURES / 'gradcam_examples.png', width=13)
attr = pd.read_csv(RESULTS / 'baseline/per_image_attributions.csv')
attr.groupby('group_id')['background_attribution_ratio'].describe()

## 4. RQ2: background reliance vs performance

In [ ]:
with open(RESULTS / 'baseline/rq2/rq2_analysis.json') as f:
    rq2 = json.load(f)
{k: v for k, v in rq2.items() if k != 'group_stats'}

In [ ]:
show(FIGURES / 'background_reliance.png', width=7)
show(FIGURES / 'attribution_by_group.png', width=11)
show(FIGURES / 'attribution_vs_confidence.png', width=7)

## 5. XAI-guided targeted augmentation (Experiment D / RQ3)

The top 25% most background-reliant **training** images (by baseline Grad-CAM ratio) receive stronger background-altering augmentation.

In [ ]:
with open(RESULTS / 'baseline/selection.json') as f:
    print(json.load(f))

hist = pd.read_csv(RESULTS / 'xai_guided/history.csv')
ax = hist.plot(x='epoch', y=['train_loss', 'val_loss'], marker='o')
ax.set_title('XAI-guided loss'); plt.show()

In [ ]:
comparison = pd.read_csv(RESULTS / 'metrics_comparison.csv')
comparison

show(FIGURES / 'baseline_vs_modified.png', width=14)

## 6. Takeaways

See `report/experimental_summary.md` for the full discussion, including limitations
(bounding boxes are an approximation of the bird region; association is not causation;
single seed, short training schedule).